# UPDATE_DB
Run this daily (or every few days) to refresh the local price cache. Fetches full history for every ticker across SP500/DJIA/NDX/TSX60, tags each row with which index(es) it belongs to, and overwrites `Data Input/prices.parquet`. Prototype-grade: full refetch + full overwrite every run, no incremental fetch, no upsert, no point-in-time membership tracking.

In [1]:
import importlib
import pandas as pd

import momentum_fetch as fetch
import momentum_backend as backend

importlib.reload(fetch)
importlib.reload(backend)

<module 'momentum_backend' from 'c:\\Users\\User\\Desktop\\MOMENTUM_NEW\\momentum_backend.py'>

## Build the universe + membership map

In [2]:
INDEX_SPECS = [
    ("sp500", "US", "sp500"),
    ("djia",  "US", "djia"),
    ("ndx",   "US", "ndx"),
    ("tsx60", "CAN", "tsx60"),
]

membership = {}
for label, market, universe in INDEX_SPECS:
    tickers = fetch.get_universe_tickers_simple(market, universe)
    for t in tickers:
        membership.setdefault(t, set()).add(label)
    print(f"{label}: {len(tickers)} tickers")

all_tickers = sorted(membership.keys())
print(f"\nTotal unique tickers across all indices: {len(all_tickers)}")

sp500: 503 tickers
djia: 30 tickers
ndx: 101 tickers
tsx60: 60 tickers

Total unique tickers across all indices: 578


## Fetch full history
`keep_survivors=False` here on purpose — this is a persistent cache, not a one-off backtest input. Dropping a ticker's whole history just because it's not trading *today* is exactly the opposite of what a cache should do; we want to keep whatever we successfully fetched, delisted or not.

In [3]:
adj_close, tickers_success, tickers_failed = backend.fetch_prices(
    all_tickers, keep_survivors=False, ffill_prices=True
)
print(f"fetched {len(tickers_success)} tickers OK, {len(tickers_failed)} failed")
if tickers_failed:
    print("failed:", tickers_failed)

  0%|          | 0/578 [00:00<?, ?it/s]

fetched 578 tickers OK, 0 failed


## Reshape to long format + tag membership

In [4]:
long_df = adj_close.reset_index().melt(id_vars=adj_close.index.name or "index", var_name="ticker", value_name="adj_close")
long_df = long_df.rename(columns={long_df.columns[0]: "date"})
long_df = long_df.dropna(subset=["adj_close"])

index_labels = [label for label, _, _ in INDEX_SPECS]
for label in index_labels:
    tickers_in_index = {t for t, labels in membership.items() if label in labels}
    long_df[f"is_{label}"] = long_df["ticker"].isin(tickers_in_index)

print(long_df.shape)
long_df.head()

C:\Users\User\AppData\Local\Temp\ipykernel_8556\3174127829.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long_df = adj_close.reset_index().melt(id_vars=adj_close.index.name or "index", var_name="ticker", value_name="adj_close")


(4928568, 7)


,date,ticker,adj_close,is_sp500,is_djia,is_ndx,is_tsx60
9615,1999-11-18,A,26.189775,True,False,False,False
9616,1999-11-19,A,24.032093,True,False,False,False
9617,1999-11-22,A,26.189775,True,False,False,False
9618,1999-11-23,A,23.808878,True,False,False,False
9619,1999-11-24,A,24.441311,True,False,False,False


## Write the cache

In [6]:
out_path = r"Data Input\prices.parquet"
long_df.to_parquet(out_path, index=False)
print(f"wrote {out_path}: {len(long_df):,} rows, {long_df['ticker'].nunique()} tickers, "
      f"{long_df['date'].min().date()} to {long_df['date'].max().date()}")

wrote Data Input\prices.parquet: 4,928,568 rows, 578 tickers, 1962-01-02 to 2026-09-29
